In [ ]:
import numpy as np
import pandas as pd

import calvados as cal # https://github.com/KULL-Centre/CALVADOS
import matplotlib.pyplot as plt
import seaborn as sns
import openmm

In [ ]:
plt.rcParams['font.size'] = 6
plt.rcParams['ytick.major.size'] = 2
plt.rcParams['ytick.minor.size'] = 1
plt.rcParams['xtick.major.size'] = 2
plt.rcParams['xtick.minor.size'] = 1
plt.rcParams['xtick.labelsize'] = 6
plt.rcParams['ytick.labelsize'] = 6
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['axes.labelpad'] = 2
plt.rcParams['savefig.pad_inches'] = 0.1
plt.rcParams['figure.dpi'] = 300
plt.rcParams['lines.markersize'] = 3
plt.rcParams['lines.markeredgewidth'] = 0.5
plt.rcParams['lines.linewidth'] = 1.
plt.rcParams['font.serif'] = 'Times New Roman'
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.monospace'] = 'Courier New'

# Load dataframes

In [ ]:
df_rg = pd.read_csv('../parameterisation/mdp_rg.csv')
df_pae1 = pd.read_csv('../parameterisation/mdp_df_30.csv') # PAE-Param
df_zn = pd.read_csv('../parameterisation/mdp_zn.csv')

In [ ]:
df_peptone = pd.read_csv('PeptoneDB-SAXS_with_temp_ionic.csv')
pdb_folder_peptone = 'pdbs_sasdb' # point towards sasdb pdbs
df_peptone['N'] = df_peptone['length']
df_peptone.rename({'label': 'uniprot'},axis=1,inplace=True)

In [ ]:
df_cyto = pd.read_csv('../cytosol/mdp_cyto.csv')
df_tf = pd.read_csv('../cytosol/mdp_tf.csv')
df_pae2 = pd.read_csv('../cytosol/pae2_df.csv')

# Collect data

In [ ]:
dfs = {
    'Cao' : df_rg,
    'PAE-Param' : df_pae1,
    'Zn-Finger' : df_zn,
    'Intracell' : df_cyto,
    'TF' : df_tf,
    'PAE-Test' : df_pae2,
    'PeptoneDB-SAXS' : df_peptone,
}

In [ ]:
for name, df in dfs.items():
    if 'Unnamed: 0' in df.columns:
        df.drop('Unnamed: 0', axis=1, inplace=True)

# pLDDT and DSSP

In [ ]:
def calc_ss(pdb_folder, dssp_folder, df):
    loops = []
    helixs = []
    betas = []
    
    for key, val in df.iterrows():
        name = val['uniprot']
        pdb = f'{pdb_folder}/{name}.pdb'
        bfac = cal.build.bfac_from_pdb(pdb,confidence=0.)
        df.loc[key,'plddt'] = np.mean(bfac)*100

        loop = 0
        helix = 0
        beta = 0
        with open(f'{dssp_folder}/{name}.pdb.dssp','r') as f:
            # print(len(list(f.readlines())))
            # print(len(bfac))
            for bf, line in zip(bfac,f.readlines()):
                if bf < 0.7:
                    continue
                spl = line.split()
                ss = spl[2]
                if ss in ['H','G','I']:
                    helix += 1
                elif ss in ['B','E']:
                    beta += 1
                else:
                    loop += 1
            ct = helix + beta + loop
            if ct > 0.:
                helix /= ct
                beta /= ct
                loop /= ct
        helixs.append(helix)
        betas.append(beta)
        loops.append(loop)
    helixs = np.array(helixs)
    betas = np.array(betas)
    loops = np.array(loops)
    return helixs, betas, loops

In [ ]:
dssp_folder = 'dssp_results' # Results from biopython dssp runs, zenodo
pdb_folder = 'pdbs_combined' # This requires pooling all pdbs from all datasets into one folder

for name, df in dfs.items():
    print(name)
    helix, beta, loop = calc_ss(pdb_folder, dssp_folder, df)
    df['helix'] = helix*100
    df['beta'] = beta*100
    df['loop'] = loop*100

In [ ]:
for name, df in dfs.items():
    df_nonzero = df.loc[df['helix'] + df['beta'] + df['loop'] > 0.]
    print(f'{name} & {len(df)} & {np.mean(df["N"]):.1f} & {np.mean(df["plddt"]):.1f} & {np.mean(df_nonzero["helix"]):.1f} & {np.mean(df_nonzero["beta"]):.1f} & {np.mean(df_nonzero["loop"]):.1f} \\\\')

In [ ]:
features = ['N','plddt','helix','beta','loop']
labels = {
    'N': 'N residues',
    'plddt' : 'pLDDT',
    'helix' : '% Helix \n (pLDDT>70)',
    'beta' : '% Extended \n (pLDDT>70)',
    'loop' : '% Loop \n (pLDDT>70)'
}

fig, ax = plt.subplots(1,len(features),figsize=(8,1.5))
for idx, feat in enumerate(features):
    axij = ax[idx]
    for jdx, (name, df) in enumerate(dfs.items()):
        color = f'C{jdx}'
        if idx == 0:
            label = name
        else:
            label = None
        if len(df) < 5:
            for kdx, x in enumerate(df[feat].values):
                if kdx > 0:
                    label = None
                axij.axvline(x,color=color,label=label,lw=0.8)
        else:
            sns.kdeplot(df[feat].values,ax=axij,color=color,label=name,bw_adjust=0.5,lw=0.8)#,fill=True)
    axij.set_xlabel(labels[feat])
    axij.set_ylabel('pdf')
    axij.grid(False)
ax[0].legend(fontsize=4)
fig.tight_layout()
# fig.savefig('histograms_datasets.pdf')

# Performance analysis

In [ ]:
path = 'sims_performance' # Path to sims_performance from zenodo

methods = ['idp', 'calvados3', 'af_calvados']

for key, val in df_rg.iterrows():
    name = val['uniprot']

    for method in methods:
        with open(f'{path}/{method}/{name}/{name}.log','r') as f:
            last_line = f.readlines()[-1]
            spl = last_line.split()
            perf = float(spl[1])
        df_rg.loc[key, f'perf_{method}'] = perf

        system = openmm.XmlSerializer.deserialize(open(f'{path}/{method}/{name}/{name}.xml').read())
        if method == 'idp':
            df_rg.loc[key, f'nrestr_{method}'] = 0
        elif method == 'calvados3':
            nrestr = system.getForces()[3].getNumBonds()
            df_rg.loc[key, f'nrestr_{method}'] = nrestr
        elif method == 'af_calvados':
            nrestr = 0
            for idx in range(3,6):
                nrestr += system.getForces()[idx].getNumBonds()
            df_rg.loc[key, f'nrestr_{method}'] = nrestr

In [ ]:
labels = ['IDP','CALVADOS 3','AF-CALVADOS']

c3_color = 'skyblue'
cgo_color = 'tab:green'

fig, ax = plt.subplots(1,3,figsize=(7,2))

axij = ax[0]
axij.plot(df_rg[f'nrestr_calvados3'], df_rg[f'nrestr_af_calvados'],'o',c='black')
xs = np.arange(0,3301)
axij.set(xlim=(0,3300),ylim=(0,3300))
axij.plot(xs,xs,ls='dotted',color='black')
axij.set(xlabel='$N_\mathrm{restraints}\,$(CALVADOS 3)', ylabel='$N_\mathrm{restraints}\,$(AF-CALVADOS)')

axij = ax[1]
axij.plot(df_rg[f'perf_idp']/1000, df_rg[f'perf_calvados3']/1000,'o',c=c3_color,label=labels[1])
axij.plot(df_rg[f'perf_idp']/1000, df_rg[f'perf_af_calvados']/1000,'o',c=cgo_color,label=labels[2])
xs = np.arange(20,34,0.1)
axij.plot(xs,xs,ls='dotted',color='black')
axij.set(xlim=(20,33),ylim=(20,33))
axij.set(xlabel='Performance (no restraints) [$\mu$s/day]', ylabel='Performance (restraints) [$\mu$s/day]')
axij.legend(fontsize=5)

axij = ax[2]
axij.plot(df_rg[f'perf_calvados3']/1000, df_rg[f'perf_af_calvados']/1000,'o',c='black')
xs = np.arange(20,34,0.1)
axij.plot(xs,xs,ls='dotted',color='black')
axij.set(xlim=(20,33),ylim=(20,33))
axij.set(xlabel='Performance (CALVADOS 3) [$\mu$s/day]', ylabel='Performance (AF-CALVADOS) [$\mu$s/day]')

fig.tight_layout()
# fig.savefig('performance.pdf')

print(np.mean(df_rg[f'perf_calvados3'] / df_rg[f'perf_idp']))
print(np.mean(df_rg[f'perf_af_calvados'] / df_rg[f'perf_idp']))
print('-'*30)
print(np.mean(df_rg[f'perf_af_calvados'] / df_rg[f'perf_calvados3']))